# AI Usage Protocol — M3-critical-evaluation

**Student:** nnaemeka-nnachi-egwu  
**Course:** Real-Time Systems Seminar — HSHL, Prof. Dr. Stefan Henkler  
**Topic:** Priority Inheritance Protocol (Resource Access Protocols)  
**Exported:** 2026-06-03T17:30:00+02:00  
**Total entries:** 10  

This notebook documents all significant AI/LLM interactions during M3-critical-evaluation.  
JSON: `ai_protocol_m3.json`


In [ ]:
import json
with open('ai_protocol_m3.json', 'r', encoding='utf-8') as f:
    protocol = json.load(f)
print(f"Milestone: {protocol['milestone']}")
print(f"Entries: {len(protocol['entries'])}")
print(f"Protocol version: {protocol['protocol_version']}")

In [ ]:
print(f"{'ID':<16} {'Status':<22} {'Use':<8} {'Section':<18} {'Objective'}")
print('-'*110)
for entry in protocol['entries']:
    print(f"{entry['id']:<16} {entry['evaluation']['status']:<22} "
          f"{entry['evaluation']['usefulness']:<8} "
          f"{entry['integration']['section']:<18} "
          f"{entry['objective'][:55]}")

---

### `e-m3-001` — Construct a minimal deadlock counterexample for PIP

| | |
|---|---|
| **Timestamp** | 2026-06-03T09:00:00+02:00 |
| **Parent** | — |
| **Tool / Model** | Claude / claude-sonnet-4-6 |
| **Verification level** | `primary-source-check` |
| **Evaluation status** | `accepted` |
| **Usefulness** | high |
| **Used in submitted work** | True |
| **Section** | M3 §2.L1 |
| **Usage type** | comparison-structuring |
| **Direct text reused** | False |

**Prompt:**
> buttazzo figure 7.13 shows a deadlock under PIP. can you give me the minimal counterexample — the smallest scenario that demonstrates it — so i can include it in M3

**AI Output Summary:**
Two tasks (tau1 high, tau2 low), two semaphores (Sa, Sb). tau1: wait(Sa);wait(Sb). tau2: wait(Sb);wait(Sa). At t1 tau2 locks Sb, t2 tau1 preempts, t3 tau1 locks Sa, t4 tau1 tries Sb -> blocked, tau2 inherits P1, t5 tau2 tries Sa -> deadlock.

**Verification actions:**
  - Traced exactly against Buttazzo Fig.7.13 (p.226) and surrounding text.
  - Confirmed: 'the deadlock does not depend on the Priority Inheritance Protocol but is caused by an erroneous use of semaphores' (p.227).
  - Verified that PIP's pi_wait does not check for lock cycles — it only does priority inheritance, not cycle detection.
  - Fix: total ordering on semaphore acquisition (by semaphore ID) prevents the cycle.

**Issues / Corrections:**
  - (none)

**Reflection:**
The counterexample is exactly as in Buttazzo. Important insight: PIP's architectural limitation (no cycle detection) is not a bug but a design choice for simplicity and transparency. PCP adds the lock-grant rule that prevents cycles. This trade-off is worth discussing in M3.


---

### `e-m3-002` — Construct the minimal chained blocking counterexample for PIP

| | |
|---|---|
| **Timestamp** | 2026-06-03T10:15:00+02:00 |
| **Parent** | — |
| **Tool / Model** | Claude / claude-sonnet-4-6 |
| **Verification level** | `primary-source-check` |
| **Evaluation status** | `partially-accepted` |
| **Usefulness** | high |
| **Used in submitted work** | True |
| **Section** | M3 §2.L2 |
| **Usage type** | comparison-structuring |
| **Direct text reused** | False |

**Prompt:**
> i need the minimal chained blocking example for PIP — buttazzo figure 7.12. what is the exact scenario with 3 tasks and 2 semaphores?

**AI Output Summary:**
Three tasks (tau1>tau2>tau3), two semaphores Sa Sb. tau1 needs Sa then Sb sequentially. tau2 holds Sb in CS. tau3 holds Sa in CS. When tau1 arrives: first blocked on Sa (direct blocking), then on Sb (second blocking). Two separate blocking events: chained.

**Verification actions:**
  - Traced against Buttazzo Fig.7.12 (p.225) and §7.6.5 description.
  - Verified: 'In the worst case, if tau1 accesses n distinct semaphores... tau1 will be blocked for the duration of n critical sections' (p.225).
  - Confirmed this is exactly what Theorem 7.2 bounds: alpha_i = min(l_i, s_i) = min(1,2)=1... wait — need to check with both sems.
  - With 2 semaphores both potentially blocking tau1: l_1=2, s_1=2, alpha_1=2. Correct.

**Issues / Corrections:**
  - Initially confused about whether alpha_1=1 or alpha_1=2 in the 2-semaphore chained case. Rechecked: l_1=2 (tau2 and tau3 both block tau1), s_1=2 (Sa and Sb both block tau1), so alpha_1=min(2,2)=2. The bound says tau1 can be blocked for 2 CS in this scenario.

**Reflection:**
Had to correct my initial reading of alpha_i in the chained case. For the minimal counterexample l_1=s_1=2 so alpha_1=2, which is exactly the bound hit. Good check that Theorem 7.2 is tight in this case even though Eq 7.11 is generally pessimistic.


---

### `e-m3-003` — Evaluate whether the blocking time bound is tight or pessimistic, rela

| | |
|---|---|
| **Timestamp** | 2026-06-03T11:30:00+02:00 |
| **Parent** | e-m3-001 |
| **Tool / Model** | Claude / claude-sonnet-4-6 |
| **Verification level** | `methodological-check` |
| **Evaluation status** | `accepted` |
| **Usefulness** | high |
| **Used in submitted work** | True |
| **Section** | M3 §3 |
| **Usage type** | argument-critique |
| **Direct text reused** | False |

**Prompt:**
> buttazzo says the Eq 7.11 bound is not tight (p.222). how tight is it in practice? and how does the WCET estimation problem from the Abella 2015 paper make this worse?

**AI Output Summary:**
The bound is not tight because Bl can sum two CS from the same semaphore (impossible per Lemma 7.4). Rajkumar's exact algorithm is exponential. In practice the gap depends on the task set. The WCET problem compounds this: if delta_{j,k} is overestimated (conservative WCET), B_i is overestimated too, causing false rejections in schedulability tests.

**Verification actions:**
  - Verified Buttazzo p.222: 'the blocking factors derived by this algorithm are not tight'.
  - WCET paper conclusion (10.txt): 'none of the surveyed methods is fully trustworthy on all accounts'.
  - Identified two pessimism sources: (1) Eq 7.11 structural pessimism, (2) conservative delta_{j,k} from WCET analysis.
  - Noted direction: if WCET tools produce OVER-estimates of delta, tasks may be unnecessarily rejected. If UNDER-estimates, system appears schedulable but is not (more dangerous).

**Issues / Corrections:**
  - (none)

**Reflection:**
Two distinct sources of pessimism stacked on top of each other. The paper can distinguish them clearly: structural pessimism from Eq 7.11 (Buttazzo's own admission) and input uncertainty from WCET analysis (WCET paper). Together they motivate both the extension to tighter algorithms AND better WCET tooling.


---

### `e-m3-004` — Identify concrete suitable application scenarios for PIP in embedded s

| | |
|---|---|
| **Timestamp** | 2026-06-03T13:00:00+02:00 |
| **Parent** | — |
| **Tool / Model** | Claude / claude-sonnet-4-6 |
| **Verification level** | `methodological-check` |
| **Evaluation status** | `accepted` |
| **Usefulness** | medium |
| **Used in submitted work** | True |
| **Section** | M3 §5 |
| **Usage type** | idea-generation |
| **Direct text reused** | False |

**Prompt:**
> give me concrete scenarios where PIP is the RIGHT choice — not just 'when deadlock is not possible' but specific embedded system examples

**AI Output Summary:**
1. Automotive ECU (single-core): OSEK task set with 2-3 shared data buffers, total semaphore ordering enforced, short critical sections. 2. POSIX RTOS legacy migration: existing pthread code with mutexes, no ceiling declarations possible. 3. Simple industrial PLC with 1 shared sensor buffer accessed by control and monitoring tasks.

**Verification actions:**
  - Verified each scenario against PIP requirements: single processor, fixed priority (RM/DM), no required deadlock prevention (or deadlock provably absent), legacy code or no ceiling info.
  - Automotive ECU: consistent with OSEK discussion in M2. Single-core assumption holds for classic AUTOSAR.
  - POSIX migration: consistent with transparency argument from Buttazzo §7.10.
  - PLC scenario: plausible but not directly cited — marked as illustrative.

**Issues / Corrections:**
  - (none)

**Reflection:**
Good concrete grounding for the transfer analysis. The automotive ECU case is the strongest because it connects to real standards (OSEK/AUTOSAR) and is consistent with earlier M2 content. The POSIX migration case demonstrates the transparency argument from Buttazzo §7.10.


---

### `e-m3-005` — Identify concrete unsuitable scenarios for PIP — where PIP specificall

| | |
|---|---|
| **Timestamp** | 2026-06-03T14:30:00+02:00 |
| **Parent** | e-m3-004 |
| **Tool / Model** | Claude / claude-sonnet-4-6 |
| **Verification level** | `methodological-check` |
| **Evaluation status** | `accepted` |
| **Usefulness** | high |
| **Used in submitted work** | True |
| **Section** | M3 §5 |
| **Usage type** | argument-critique |
| **Direct text reused** | False |

**Prompt:**
> now give me concrete failure scenarios for PIP — not just multiprocessor is hard but specific cases where using PIP would cause real problems

**AI Output Summary:**
1. Two mutexes in reverse order without ordering enforcement -> deadlock (Fig 7.13 scenario). 2. Multi-core SoC with task migration -> IPI latency breaks inheritance timing. 3. EDF-scheduled control system -> PIP not defined for dynamic priority. 4. System with deep nested CS chains -> transitive traversal adds kernel interrupt latency.

**Verification actions:**
  - Scenario 1: directly from Buttazzo §7.6.5 Fig 7.13.
  - Scenario 2: grounded in Gracioli [GP] overhead model.
  - Scenario 3: footnote 2, Buttazzo p.209 confirms PIP extension for EDF exists (Spuri) but is not in Ch.7.
  - Scenario 4: Buttazzo §7.6.4 p.224 'the transitivity rule is applied' — unbounded chain traversal.

**Issues / Corrections:**
  - (none)

**Reflection:**
Scenario 4 (deep nesting) is underappreciated in many PIP discussions. The kernel traversal during pi_wait is a real-time concern: it adds latency inside an atomic operation. Linux RTMUTEX_DEPTH_LIMIT=48 shows that even mature implementations bound the chain depth — which means the formal guarantees may not fully hold on the implementation.


---

### `e-m3-006` — Propose concrete achievable extensions — particularly POSIX validation

| | |
|---|---|
| **Timestamp** | 2026-06-03T16:00:00+02:00 |
| **Parent** | — |
| **Tool / Model** | Claude / claude-sonnet-4-6 |
| **Verification level** | `plausibility-check` |
| **Evaluation status** | `accepted` |
| **Usefulness** | high |
| **Used in submitted work** | True |
| **Section** | M3 §6 |
| **Usage type** | idea-generation |
| **Direct text reused** | False |

**Prompt:**
> what are realistic extensions i could actually implement or run for this seminar? not vague future work

**AI Output Summary:**
1. POSIX validation: pthread + SCHED_FIFO + PTHREAD_PRIO_INHERIT, measure blocking with clock_gettime. 2. UPPAAL: extend model to nested CS, verify Lemma 7.2 (transitivity only with nesting), verify deadlock with reverse semaphore order. 3. Python exact blocking: implement Rajkumar exhaustive search for n<=8, compare with Eq 7.11.

**Verification actions:**
  - POSIX experiment: feasible on any Linux system with real-time patch or kernel >= 2.6.25.
  - UPPAAL extension: feasible within the model already being built — need second semaphore.
  - Python exhaustive search: feasible for small n, O(2^n) but n<=8 is practical.
  - All three: achievable within seminar timeframe.

**Issues / Corrections:**
  - (none)

**Reflection:**
These are concrete enough to actually do. The POSIX experiment in particular would let me report real measurements alongside theoretical B_i values — exactly the kind of empirical validation the M3 checklist asks for ('I evaluated applicability to realistic embedded systems scenarios').


---

### `e-m3-007` — Critical check: is the M3 evaluation balanced (not just criticism)?

| | |
|---|---|
| **Timestamp** | 2026-06-03T17:00:00+02:00 |
| **Parent** | — |
| **Tool / Model** | Claude / claude-sonnet-4-6 |
| **Verification level** | `methodological-check` |
| **Evaluation status** | `accepted` |
| **Usefulness** | medium |
| **Used in submitted work** | True |
| **Section** | M3 |
| **Usage type** | argument-critique |
| **Direct text reused** | False |

**Prompt:**
> looking at my M3 draft — is it too negative about PIP? the checklist says identify strengths too. am i balanced?

**AI Output Summary:**
Current draft has 5 strength points (S1-S5) and 3 limitation points (L1-L3). L3 (transitive complexity) is arguably the least severe in practice. The strengths (Theorem 7.2 bound, transparency, low pessimism, feed into schedulability) are concrete and sourced. Balance seems reasonable for a critical evaluation.

**Verification actions:**
  - Re-read M3 §1 and §2. Strengths section has 5 specific points with Buttazzo citations.
  - Limitations section has exact counterexamples (not vague criticisms).
  - Transfer analysis (§5) has concrete suitable AND unsuitable scenarios.
  - Checklist item: 'I identified concrete strengths, not only generic advantages' — satisfied.

**Issues / Corrections:**
  - (none)

**Reflection:**
The balance is reasonable. PIP is genuinely a useful protocol with real limitations. Calling it 'hard' to implement (Table 7.5) while also calling it 'transparent' (§7.10) is not a contradiction — hard for the kernel implementer, transparent for the application programmer. That nuance is worth making explicit in M3 and M4.


---

### `e-m3-008` — Critically examine whether the 'transparency' strength is as strong as

| | |
|---|---|
| **Timestamp** | 2026-06-04T09:00:00+02:00 |
| **Parent** | — |
| **Tool / Model** | Claude / claude-sonnet-4-6 |
| **Verification level** | `methodological-check` |
| **Evaluation status** | `accepted` |
| **Usefulness** | high |
| **Used in submitted work** | True |
| **Section** | M3 §5, M4 §VII |
| **Usage type** | argument-critique |
| **Direct text reused** | False |

**Prompt:**
> i listed transparency as a strength of pip. but is it really a strength if it makes PIP harder to audit? with HLP you can see from the code exactly which semaphore a task will raise its priority for. with PIP that's invisible.

**AI Output Summary:**
That's a valid critique. Transparency has two sides: (1) migration ease — legacy code works without changes, and (2) reduced auditability — the priority boost is invisible at the call site. For safety-critical certification (DO-178C, ISO 26262), auditability matters: you need to trace every priority elevation to its source. PCP/HLP's explicit ceiling declarations make the priority behaviour statically analysable. PIP's dynamic inheritance requires tracing execution history to understand why a task ran at a given priority.

**Verification actions:**
  - This is a genuine engineering trade-off not explicitly discussed in Buttazzo but consistent with the certification requirements literature.
  - Buttazzo does say transparency means 'can be implemented without modifying task code' (§7.10 p.247) — doesn't mention the audit implication.
  - The audit point is my own analysis, flagged as such in M3. Not directly from Buttazzo.

**Issues / Corrections:**
  - (none)

**Reflection:**
Added a nuance to the transparency 'strength': it's a strength for migration but a potential weakness for safety-critical certification where static auditability is required. This is the kind of two-sided analysis the M3 checklist asks for — not just listing advantages but examining the assumptions under which they hold.


---

### `e-m3-009` — Check whether the deadlock counterexample requires exactly two tasks o

| | |
|---|---|
| **Timestamp** | 2026-06-04T11:30:00+02:00 |
| **Parent** | e-m3-001 |
| **Tool / Model** | Claude / claude-sonnet-4-6 |
| **Verification level** | `primary-source-check` |
| **Evaluation status** | `accepted` |
| **Usefulness** | medium |
| **Used in submitted work** | True |
| **Section** | M3 §2.L1 |
| **Usage type** | argument-critique |
| **Direct text reused** | False |

**Prompt:**
> the deadlock counterexample has two tasks and two semaphores. does the deadlock require exactly two tasks or can it happen with more? does PIP make it more or less likely?

**AI Output Summary:**
A deadlock cycle can involve any number of tasks (n tasks, n semaphores in a cycle). Buttazzo shows n=2 as the minimal case (Fig 7.13). PIP does not increase deadlock probability — the deadlock comes from the lock ORDER chosen by the programmer, not from PIP itself. In fact, Buttazzo explicitly says 'the deadlock does not depend on the Protocol but is caused by erroneous use of semaphores' (p.227). PIP is neutral regarding deadlock; PCP actively prevents it via the lock-grant rule.

**Verification actions:**
  - Verified Buttazzo p.227: 'the deadlock does not depend on the Priority Inheritance Protocol but is caused by an erroneous use of semaphores.'
  - Verified: the circular wait condition for deadlock is independent of PIP — it's a property of the lock acquisition order.
  - Checked Theorem 7.3 in PCP section: PCP prevents deadlock via the ceiling rule — no circular wait can form.

**Issues / Corrections:**
  - (none)

**Reflection:**
Important clarification: PIP doesn't CAUSE deadlocks, it just doesn't PREVENT them. This distinction matters for the critical evaluation — I shouldn't write 'PIP causes deadlock risk' but rather 'PIP provides no protection against deadlocks that arise from cyclic semaphore acquisition orders.' The same deadlocks would occur without ANY resource access protocol.


---

### `e-m3-010` — Assess whether the WCET trustworthiness issue applies equally to ALL p

| | |
|---|---|
| **Timestamp** | 2026-06-04T15:00:00+02:00 |
| **Parent** | — |
| **Tool / Model** | Claude / claude-sonnet-4-6 |
| **Verification level** | `methodological-check` |
| **Evaluation status** | `accepted` |
| **Usefulness** | high |
| **Used in submitted work** | True |
| **Section** | M3 §3, M4 §VII |
| **Usage type** | argument-critique |
| **Direct text reused** | False |

**Prompt:**
> i used the abella 2015 WCET paper as a critique of pip's blocking computation. but doesn't the same WCET problem apply to NPP, HLP, and PCP too? is this critique specific to PIP?

**AI Output Summary:**
You're right that WCET estimation is a general problem affecting all protocols. However, the critique has different severity depending on the protocol. NPP and HLP use the longest CS of any lower task — a single maximum, easy to identify. PIP uses a sum over multiple CS weighted by a complex condition (Eq 7.11) — more terms means more WCET inputs, each with its own uncertainty. PCP has the same issue. SRP too. So the WCET critique isn't unique to PIP, but the number of uncertain inputs is protocol-dependent.

**Verification actions:**
  - Verified: Buttazzo §7.4 NPP formula (Eq 7.3) uses a single maximum δ — one WCET input per blocking bound.
  - Verified: Buttazzo §7.6.3 PIP formula (Eqs 7.9-7.11) sums multiple δ values — more inputs.
  - Decision: reframe the M3 critique as 'the blocking bound formula requires multiple WCET inputs, each subject to estimation uncertainty' rather than 'WCET is a problem for PIP specifically.'

**Issues / Corrections:**
  - (none)

**Reflection:**
Good self-correction. The WCET critique is general but has protocol-specific weight depending on how many CS durations appear in the blocking formula. For NPP it's one number; for PIP/PCP it's a sum over multiple CS. I've revised M3 to make this distinction explicit rather than implying WCET is uniquely a PIP problem.


In [ ]:
import re

def _sb(t):
    t=(t or 'ai').lower()
    return re.sub(r'[^a-z0-9]+','-',t).strip('-') or 'ai'

def _eb(v):
    v=str(v or '')
    for a,b in [('\\','\\textbackslash{}'),'{','\\{'),'}','\\}'),'&','\\&'),'%','\\%'),'$','\\$'),'#','\\#'),'_','\\_')]:
        v=v.replace(a,b)
    return v

p=protocol
year=p.get('exported_at','2026')[:4]
key='-'.join(['ai-usage-protocol',_sb(p.get('student_id','')),_sb(p.get('milestone','')),year])
note=(f"Machine-readable AI protocol; course: {p.get('course','')}; "
      f"version: {p.get('protocol_version','0.2')}; "
      f"entries: {len(p.get('entries',[])) }; json: ai_protocol_m3.json")
bib=(f'@misc{{{key},\n'
     f'  author = {{{_eb(p.get("student_id",""))}}},\n'
     f'  title = {{{_eb(f"AI Usage Protocol for {p.get(chr(109)+chr(105)+chr(108)+chr(101)+chr(115)+chr(116)+chr(111)+chr(110)+chr(101),\"\")}: {p.get(chr(116)+chr(111)+chr(112)+chr(105)+chr(99),\"\")}")}}},\n'
     f'  year = {{{year}}},\n'
     f'  howpublished = {{{{Submitted AI usage protocol notebook and JSON file}}}},\n'
     f'  note = {{{_eb(note)}}}\n}}'
)
print(bib)